In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

import TutorialFeedback from '@site/src/components/TutorialFeedback';





*Tantiya sa paggamit: 37 minuto sa isang Heron processor (PAALALA: Ito ay isang tantiya lamang. Maaaring mag-iba ang iyong runtime.)*

## Panimula

Ang tutorial na ito ay nagpapakita kung paano bumuo, mag-deploy, at magpatakbo ng isang development workflow na tinatawag na [Qiskit pattern](/guides/intro-to-patterns) para sa pagsisimula ng Heisenberg chain at pagtatantya ng ground-state energy nito gamit ang SPSA optimizer.

## Mga Kinakailangan

Bago magsimula sa tutorial na ito, tiyaking mayroon kayong sumusunod na naka-install:

*   Qiskit SDK v2.0 o mas bago, na may suporta sa [visualization](https://docs.quantum.ibm.com/api/qiskit/visualization)
*   Qiskit Runtime v0.44 o mas bago (`pip install qiskit-ibm-runtime`)

## Pag-set up

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Sequence

from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import BaseEstimatorV2
from qiskit.circuit.library import XGate
from qiskit.circuit.library import efficient_su2
from qiskit.transpiler import PassManager
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.transpiler.passes.scheduling import (
    ALAPScheduleAnalysis,
    PadDynamicalDecoupling,
)
from qiskit_ibm_runtime import QiskitRuntimeService, Session, EstimatorV2


def visualize_results(results):
    plt.plot(results["cost_history"], lw=2)
    plt.xlabel("Number of function evaluations")
    plt.ylabel("Energy")
    plt.show()

## Hakbang 1: I-map ang classical inputs sa isang quantum problem
*   Input: Bilang ng mga spin
*   Output: Ansatz at Hamiltonian na nagmo-model ng Heisenberg chain

Bumuo ng ansatz at Hamiltonian na nagmo-model ng 10-spin Heisenberg chain. Una, mag-import tayo ng ilang generic na packages at lumikha ng ilang helper functions.

In [2]:
num_spins = 10
ansatz = efficient_su2(num_qubits=num_spins, reps=2)

service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, min_num_qubits=num_spins, simulator=False
)

coupling = backend.target.build_coupling_map()
reduced_coupling = coupling.reduce(list(range(num_spins)))

edge_list = reduced_coupling.graph.edge_list()
ham_list = []

for edge in edge_list:
    ham_list.append(("ZZ", edge, 0.5))
    ham_list.append(("YY", edge, 0.5))
    ham_list.append(("XX", edge, 0.5))

for qubit in reduced_coupling.physical_qubits:
    ham_list.append(("Z", [qubit], np.random.random() * 2 - 1))

hamiltonian = SparsePauliOp.from_sparse_list(ham_list, num_qubits=num_spins)

ansatz.draw("mpl", style="iqp")

<Image src="../docs/images/tutorials/spin-chain-vqe/extracted-outputs/7e8d2f10-f1d6-4ec2-bac9-9db23499c9e1-0.avif" alt="Output of the previous code cell" />

![Output ng nakaraang code cell](../docs/images/tutorials/spin-chain-vqe/extracted-outputs/7e8d2f10-f1d6-4ec2-bac9-9db23499c9e1-0.avif)

## Hakbang 2: I-optimize ang problem para sa quantum hardware execution
*   Input: Abstract circuit, observable
*   Output: Target circuit at observable, na-optimize para sa napiling QPU

Gamitin ang `generate_preset_pass_manager` function mula sa Qiskit upang awtomatikong makagawa ng optimization routine para sa ating circuit na nauugnay sa napiling QPU. Pipiliin natin ang `optimization_level=3`, na nagbibigay ng pinakamataas na antas ng optimization sa mga preset pass managers. Isasama rin natin ang `ALAPScheduleAnalysis` at `PadDynamicalDecoupling` scheduling passes upang pigilan ang mga decoherence errors.

In [3]:
target = backend.target
pm = generate_preset_pass_manager(optimization_level=3, target=target)
pm.scheduling = PassManager(
    [
        ALAPScheduleAnalysis(durations=target.durations()),
        PadDynamicalDecoupling(
            durations=target.durations(),
            dd_sequence=[XGate(), XGate()],
            pulse_alignment=target.pulse_alignment,
        ),
    ]
)
isa_ansatz = pm.run(ansatz)
isa_observable = hamiltonian.apply_layout(isa_ansatz.layout)
isa_ansatz.draw("mpl", scale=0.6, style="iqp", fold=-1, idle_wires=False)

<Image src="../docs/images/tutorials/spin-chain-vqe/extracted-outputs/a0a5f1c8-5c31-4d9f-ae81-37bd67271d44-0.avif" alt="Output of the previous code cell" />

![Output ng nakaraang code cell](../docs/images/tutorials/spin-chain-vqe/extracted-outputs/a0a5f1c8-5c31-4d9f-ae81-37bd67271d44-0.avif)

## Hakbang 3: Magsagawa gamit ang Qiskit primitives
*   Input: Target circuit at observable
*   Output: Mga resulta ng optimization

Paliitan ang tinantyang ground-state energy ng system sa pamamagitan ng pag-optimize ng mga circuit parameters. Gamitin ang `Estimator` primitive mula sa Qiskit Runtime upang suriin ang cost function sa panahon ng optimization.

Dahil na-optimize na natin ang circuit para sa backend sa Hakbang 2, maiwasan natin ang transpilation sa Runtime server sa pamamagitan ng pagtatakda ng `skip_transpilation=True` at pagpasa ng optimized circuit. Para sa demo na ito, magpapatakbo tayo sa isang QPU gamit ang `qiskit-ibm-runtime` primitives. Upang magpatakbo gamit ang `qiskit` statevector-based primitives, palitan ang bloke ng code na gumagamit ng Qiskit Runtime primitives ng commented block.

Sa tutorial na ito, gagamitin natin ang Simultaneous Perturbation Stochastic Approximation (SPSA), na isang gradient-based optimizer. Susunod, bibigyan natin ng maikling panimula dito, at ibibigay ang code para ipatupad ang SPSA gamit ang Qiskit v2.0.
### Panimula sa SPSA
Ang Simultaneous Perturbation Stochastic Approximation (SPSA) [\[1\]](#references) ay isang optimization algorithm na nagta-tantya ng buong gradient vector gamit lamang ang dalawang function call sa bawat iteration. Hayaan ang $f:\mathbb{R}^p\rightarrow \mathbb{R}$ na maging cost function na may $p$ na mga parameter na ie-optimize, at $x_i\in \mathbb{R}^p$ ang parameter vector sa $i^{th}$ hakbang ng iteration. Upang makalkula ang gradient, isang random vector $\Delta_i$ ng laki $p$ ang nililikha, kung saan ang bawat elemento $\Delta_{ij}$, $\forall$ $j\in {1,2,...,p}$, ay uniformly na sinasample mula sa ${-1, 1}$. Susunod, ang bawat elemento ng random vector $\Delta_i$ ay pinaramihin ng maliit na halaga $c_i$ upang makagawa ng random perturbation. Ang gradient ay tinatantya bilang

$$[\nabla f(x_i)]_j \approx \frac{f(x_i + c_i \Delta_i) - f(x_i - c_i \Delta_i)}{2c_i\Delta_{ij}}.$$

Sa isang kahulugan, dahil ang random perturbation ay inilalapat sa panahon ng gradient estimation, inaasahang matatanggap at matutugunan ang maliliit na paglihis sa eksaktong mga halaga ng $f$ na nagmumula sa ingay. Sa katunayan, ang SPSA ay partikular na kilala sa pagiging matibay laban sa ingay, at nangangailangan lamang ng dalawang hardware call para sa bawat iteration. Samakatuwid, isa ito sa mga lubos na ginusting optimizer para sa pagpapatupad ng variational algorithms.

Sa tutorial na ito, ang mga hyperparameter para sa $i^{th}$ iteration, $a_i$ at $c_i$, ay kinakalkula bilang $a_i = a/(A + i + 1)^\alpha$ at $c_i = c / (i+1)^\gamma$, kung saan ang mga constant na halaga ay $A = 30$, $\alpha = 0.9$, $a = 0.3$, $c = 0.1$, at $\gamma = 0.4$. Ang mga halagang ito ay kinuha mula sa [\[2\]](#references). Ang angkop na pag-tune ng mga hyperparameter ay kinakailangan upang makuha ang magandang pagganap mula sa SPSA.

In [4]:
def spsa(
    fun, x0, args=(), A=30, alpha=0.9, a=0.3, c=0.1, gamma=0.4, maxiter=100
):
    nparams = len(x0)
    x = np.copy(x0)

    for i in range(maxiter):
        a_i = a / (A + i + 1) ** alpha
        c_i = c / (i + 1) ** gamma
        delta_i = np.random.choice([-1, 1], nparams)

        # two hardware calls
        eval_1 = fun(x + c_i * delta_i, *args)
        eval_2 = fun(x - c_i * delta_i, *args)

        # compute the gradient and update the parameters
        grad = (eval_1 - eval_2) / (2 * c_i) * np.reciprocal(delta_i)
        x = x - a_i * grad

    return x

In [8]:
def cost_func(
    params: Sequence,
    ansatz: QuantumCircuit,
    hamiltonian: SparsePauliOp,
    estimator: BaseEstimatorV2,
    cost_history_dict: dict,
) -> float:
    """Ground state energy evaluation."""
    energy = (
        estimator.run([(ansatz, hamiltonian, [params])]).result()[0].data.evs
    )

    cost_history_dict["iters"] += 1
    cost_history_dict["prev_vector"] = list(params)
    cost_history_dict["cost_history"].append(float(energy[0]))

    print(
        f"Fx Iters. done: {cost_history_dict['iters']} [Current cost: {round(energy[0], 5)}]",
        end="\r",
    )

    return energy


def solve(x0, isa_ansatz, isa_observable, maxiter=150):
    cost_history_dict = {
        "prev_vector": None,
        "iters": 0,
        "cost_history": [],
        "y_min": None,
    }

    # Evaluate the problem using a QPU via Qiskit IBM Runtime
    with Session(backend=backend) as session:
        estimator = EstimatorV2(mode=session)
        estimator.skip_transpilation = True
        estimator.options.environment.job_tags = ["TUT_HSVQE"]
        x_opt = spsa(
            cost_func,
            x0=x0,
            args=(isa_ansatz, isa_observable, estimator, cost_history_dict),
            maxiter=maxiter,
        )

        y_min = cost_func(
            x_opt, isa_ansatz, isa_observable, estimator, cost_history_dict
        )

    return y_min, cost_history_dict

In [9]:
np.random.seed(42)
num_params = ansatz.num_parameters
params = 2 * np.pi * np.random.random(num_params)

Dito itinatakda natin ang `maxiter = 50`. Tandaan na dahil ang bawat iteration ay nangangailangan ng dalawang tawag sa function upang makalkula ang gradient, ang kabuuang bilang ng mga function call ay magiging $2 \times \text{maxiter}$. Ang `maxiter` ay maaaring taasan sa anumang mas mataas na halaga para sa mas mahusay na energy estimation.

In [10]:
maxiter = 50
spsa_min, spsa_history = solve(
    params, isa_ansatz, isa_observable, maxiter=maxiter
)

Fx Iters. done: 101 [Current cost: -3.03843]

### Step 4: Post-process and return result in desired classical format

* Input: Ground-state energy estimates during optimization
* Output: Estimated ground-state energy

In [11]:
print(f"Estimated ground state energy: {spsa_min}")

Estimated ground state energy: [-3.03842968]


In [12]:
results = {
    "spsa": spsa_history,
}

visualize_results(spsa_history)

<Image src="../docs/images/tutorials/spin-chain-vqe/extracted-outputs/ecd7762a-0.avif" alt="Output of the previous code cell" />

## Large-scale hardware example

A large-scale hardware example is not included in this tutorial. As the number of qubits increases, VQE encounters significant challenges due to the [barren plateau](/learning/courses/variational-algorithm-design/optimization-loops#barren-plateaus) phenomenon: the gradient of the cost function vanishes exponentially with system size, making optimization practically infeasible for large circuits. Combined with hardware noise, this means that scaling VQE to larger spin chains does not produce reliably reproducible results. For approaches that overcome these limitations, see the Next Steps section below.

## Challenge

Now that you have a working VQE implementation for the Heisenberg chain, try the following:

1. **Experiment with ansatz depth:** Modify the `reps` parameter in `efficient_su2` (for example, try `reps=1` and `reps=3`). How does ansatz depth affect the estimated ground-state energy and convergence speed? At what point do you observe diminishing returns or instability?
2. **Tune SPSA hyperparameters:** Adjust the learning rate schedule parameters (`a`, `c`, `alpha`, `gamma`, `A`) and observe how they impact convergence. Can you find a configuration that converges faster than the defaults used here?
3. **Compare coupling topologies:** Instead of using the backend's native coupling map, try constructing a simple nearest-neighbor linear chain and compare the results. How does the connectivity of the physical hardware affect the transpiled circuit depth and final energy estimate?

## References

\[1] Spall, J. C. (2002). Implementation of the simultaneous perturbation algorithm for stochastic optimization.
IEEE Transactions on Aerospace and Electronic Systems, 34(3), 817-823.

\[2] Sahin, M. Emre, et al. (2025). Qiskit Machine Learning: an open-source library for quantum machine learning tasks at scale on quantum hardware and classical simulators. arXiv:2505.17756.

## Next steps

<Admonition type="tip" title="Recommendations">
  If you found this work interesting, you might be interested in the following material:

  * **Try Sample-based Quantum Diagonalization (SQD):** As demonstrated in this tutorial, VQE faces challenges at scale due to barren plateaus and high measurement overhead. IBM has developed [Sample-based Quantum Diagonalization (SQD)](https://www.ibm.com/quantum/blog/quantum-diagonalization) as a more scalable alternative. Unlike VQE, SQD avoids variational optimization entirely; instead, a quantum computer generates samples and a classical computer projects the Hamiltonian onto a subspace spanned by those samples and diagonalizes it. This provides an upper bound to the ground-state energy with significantly fewer measurements and without susceptibility to barren plateaus. Follow the [SQD tutorial](/docs/tutorials/sample-based-quantum-diagonalization) to see this approach in action.
  * **Explore the Quantum Diagonalization Algorithms course:** Deepen your understanding of both VQE and SQD, including their trade-offs, in the [Quantum diagonalization algorithms](/learning/courses/quantum-diagonalization-algorithms) course on IBM Quantum Learning.
</Admonition>